In [1]:
import requests
from io import StringIO
import gzip
import io
import shutil

import glob

import pandas as pd
import geopandas as gpd
from shapely.geometry import shape

---

## Microsoft Maps letöltés

https://github.com/microsoft/GlobalMLBuildingFootprints


In [2]:
def get_csv(url):
    response = requests.get(url, verify=True)
    
    if response.status_code == 200:
        return response.text
    else:
        raise ConnectionError(f"Error fetching data: {response.status_code}")

In [3]:
def download_and_extract_gz(url, output_path):
    """
    Letölti a .csv.gz fájlt egy adott URL-ről és kicsomagolja .csv formátumban (from ChatGPT)
    """
    
    gz_path = output_path + ".gz"
    response = requests.get(url, stream=True)
    
    if response.status_code == 200:
        with open(gz_path, "wb") as f:
            f.write(response.content)

        # Kicsomagolás .csv fájlba
        with gzip.open(gz_path, "rb") as f_in, open(output_path, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)

        #print(f"Fájl sikeresen letöltve és kicsomagolva: {output_path}")
    else:
        raise ConnectionError(f"Hiba a letöltés során: {response.status_code}")

---

In [27]:
dataset_links = pd.read_csv(StringIO(get_csv("https://minedbuildings.z5.web.core.windows.net/global-buildings/dataset-links.csv")))

In [28]:
hun_links = dataset_links[dataset_links.Location == 'Hungary']

In [38]:
for i, row in hun_links.iterrows():
    download_and_extract_gz(row.Url, f"data/bfp/{i}.csv")

Fájl sikeresen letöltve és kicsomagolva: data/bfp/12577.csv
Fájl sikeresen letöltve és kicsomagolva: data/bfp/12578.csv
Fájl sikeresen letöltve és kicsomagolva: data/bfp/12579.csv
Fájl sikeresen letöltve és kicsomagolva: data/bfp/12580.csv
Fájl sikeresen letöltve és kicsomagolva: data/bfp/12581.csv
Fájl sikeresen letöltve és kicsomagolva: data/bfp/12582.csv
Fájl sikeresen letöltve és kicsomagolva: data/bfp/12583.csv
Fájl sikeresen letöltve és kicsomagolva: data/bfp/12584.csv
Fájl sikeresen letöltve és kicsomagolva: data/bfp/12585.csv
Fájl sikeresen letöltve és kicsomagolva: data/bfp/12586.csv
Fájl sikeresen letöltve és kicsomagolva: data/bfp/12587.csv
Fájl sikeresen letöltve és kicsomagolva: data/bfp/12588.csv
Fájl sikeresen letöltve és kicsomagolva: data/bfp/12589.csv
Fájl sikeresen letöltve és kicsomagolva: data/bfp/12590.csv
Fájl sikeresen letöltve és kicsomagolva: data/bfp/12591.csv
Fájl sikeresen letöltve és kicsomagolva: data/bfp/12592.csv
Fájl sikeresen letöltve és kicsomagolva:

<br>

---

## csv to pandas

In [4]:
files = glob.glob("./data/bfp/*")

In [5]:
dfek = []

In [6]:
for f in files:
    df = pd.read_json(f, lines=True)
    df['geometry'] = df['geometry'].apply(shape)
    gdf = gpd.GeoDataFrame(df, crs=4326)
    dfek.append(gdf)

In [7]:
tgdf = gpd.GeoDataFrame(pd.concat(dfek, ignore_index=True))

In [8]:
tgdf

,type,properties,geometry
0,Feature,"{'height': -1.0, 'confidence': -1.0}","POLYGON ((18.98362 46.25928, 18.98369 46.25915..."
1,Feature,"{'height': -1.0, 'confidence': -1.0}","POLYGON ((18.98312 46.18515, 18.98316 46.18516..."
2,Feature,"{'height': -1.0, 'confidence': -1.0}","POLYGON ((18.9837 46.53721, 18.9837 46.53714, ..."
3,Feature,"{'height': -1.0, 'confidence': -1.0}","POLYGON ((18.98338 46.53572, 18.98338 46.53566..."
4,Feature,"{'height': -1.0, 'confidence': -1.0}","POLYGON ((18.98418 46.52752, 18.9842 46.52756,..."
...,...,...,...
5685953,Feature,"{'height': -1.0, 'confidence': -1.0}","POLYGON ((22.83228 48.0196, 22.83243 48.01952,..."
5685954,Feature,"{'height': -1.0, 'confidence': -1.0}","POLYGON ((22.8569 48.06529, 22.85692 48.06525,..."
5685955,Feature,"{'height': -1.0, 'confidence': -1.0}","POLYGON ((22.85755 48.06902, 22.85762 48.06903..."
5685956,Feature,"{'height': -1.0, 'confidence': -1.0}","POLYGON ((22.86218 48.04945, 22.86225 48.04947..."


In [20]:
tgdf.geometry

0          POLYGON ((18.98362 46.25928, 18.98369 46.25915...
1          POLYGON ((18.98312 46.18515, 18.98316 46.18516...
2          POLYGON ((18.9837 46.53721, 18.9837 46.53714, ...
3          POLYGON ((18.98338 46.53572, 18.98338 46.53566...
4          POLYGON ((18.98418 46.52752, 18.9842 46.52756,...
                                 ...                        
5685953    POLYGON ((22.83228 48.0196, 22.83243 48.01952,...
5685954    POLYGON ((22.8569 48.06529, 22.85692 48.06525,...
5685955    POLYGON ((22.85755 48.06902, 22.85762 48.06903...
5685956    POLYGON ((22.86218 48.04945, 22.86225 48.04947...
5685957    POLYGON ((22.86655 48.05054, 22.86667 48.0506,...
Name: geometry, Length: 5685958, dtype: geometry

In [25]:
def kozeppont(poly):
    return poly.centroid

In [26]:
tgdf["kozeppont"] = tgdf["geometry"].apply(kozeppont)

In [33]:
tgdf[["geometry", "kozeppont"]]

,geometry,kozeppont
0,"POLYGON ((18.98362 46.25928, 18.98369 46.25915...",POINT (18.98371 46.25923)
1,"POLYGON ((18.98312 46.18515, 18.98316 46.18516...",POINT (18.98314 46.18517)
2,"POLYGON ((18.9837 46.53721, 18.9837 46.53714, ...",POINT (18.98374 46.53718)
3,"POLYGON ((18.98338 46.53572, 18.98338 46.53566...",POINT (18.98344 46.53569)
4,"POLYGON ((18.98418 46.52752, 18.9842 46.52756,...",POINT (18.98417 46.52755)
...,...,...
5685953,"POLYGON ((22.83228 48.0196, 22.83243 48.01952,...",POINT (22.83239 48.01959)
5685954,"POLYGON ((22.8569 48.06529, 22.85692 48.06525,...",POINT (22.85693 48.06528)
5685955,"POLYGON ((22.85755 48.06902, 22.85762 48.06903...",POINT (22.85757 48.06904)
5685956,"POLYGON ((22.86218 48.04945, 22.86225 48.04947...",POINT (22.8622 48.04947)


In [35]:
tgdf[["geometry", "kozeppont"]].reset_index().rename(columns={"index": "id"}).to_parquet("hun-teljes.parquet")